# tSCS EMG — single pulse, changed polarity + lidocaine (NTA, 24-07-2026)

## What a single-pulse file is
One **recruitment sweep**: one stimulation window per intensity (10 → 100 mA in 10 mA steps),
each EMG channel stored as a ±100 ms snippet time-locked to the pulse (t = 0).

## What is measured
- **Latency** — picked **by hand** for every muscle × intensity: the time (ms after the pulse) where
  the response starts. `NaN` = no response. This is the one manual step; picks are saved to
  `results/latency_<file>.csv` and reused from there.
- **Peak-to-peak** — from each picked latency, the window
  `[latency + OFFSET_MS, latency + OFFSET_MS + WINDOW_MS]`; p2p = max − min in it (mV).

## Files — polarity session (subject NTA, 24-07-2026, `changedpol.xlsx`)
Electrode 2, anode at the iliac crests. Every protocol was run in **both polarities** — the
**original** one and the **changed** one — before and after lidocaine (applied 45 min,
~10:50 → 11:33). Folder `testSCS`.

| | single pulse | burst | ARC-EX (Modulated) |
|---|---|---|---|
| **pre · cathodic (polarity 2)** | `100913` (10:09) | `102443` (10:24; `102236` aborted) | `103530` (10:35; `103040` no response) |
| **pre · anodic (polarity 1)** | `101941` (10:19) | `102721` (10:27) | `103849` (10:38) |
| **post · cathodic** | `113447` (11:34) | `113834` (11:38) | `114332` (11:43) |
| **post · anodic** | `113632` (11:36) | `114055` (11:40) | `114730` (11:47; `114630` ignore) |

Motor threshold from the log — burst: cathodic 25 mA pre / 30 post, anodic 30 / 30;
ARC-EX: cathodic 70 / 70, anodic 65 / 90; single pulse: ~40 pre, 30–40 post.
Participant: *"less pain with anodic"*, *"much less pain after lidocaine"*.

**Same structure as the original-polarity notebooks: before vs with lidocaine** (gray / orange),
for the polarity chosen with `POLARITY` in the config — cathodic (polarity 2, the original
montage) or anodic (polarity 1, the changed one). The last sections repeat the key figures for the
other polarity and put all four conditions together, so both lidocaine comparisons and the
polarity comparison are in one run.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import numpy as np
import matplotlib.pyplot as plt

from functions import (set_style, load_run, pretty, waterfall, waterfall_overlay,
                       latency_picker, save_latency_csv, load_latency_csv)
from functions.quantify import plot_p2p_markers
from functions.average import compare_per_muscle
from functions.compare import compare_p2p_violin, compare_latency_violin
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
FILES = {   # polarity: (before lidocaine, with lidocaine)
    "cathodic": ("Single_Pulse_autosave_20260724_100913_756ms.csv", "Single_Pulse_autosave_20260724_113447_649ms.csv"),   # polarity 2 - the original montage
    "anodic":   ("Single_Pulse_autosave_20260724_101941_811ms.csv", "Single_Pulse_autosave_20260724_113632_480ms.csv"),   # polarity 1 - the changed one
}
POLARITY = "cathodic"        # <-- polarity analysed in detail below: "cathodic" or "anodic"
OTHER    = "anodic" if POLARITY == "cathodic" else "cathodic"

LABELS = ["before lidocaine", "with lidocaine"]
CSVS   = [D + f for f in FILES[POLARITY]]

XLIM      = (-20, 80)   # ms shown in the waterfalls / picker
WINDOW_MS = 20.0        # peak-to-peak window length (ms)
OFFSET_MS = 2.0         # ...starting this long after the picked latency (ms)
YLIM_P2P  = (-0.5, 10)  # y-range for the pooled p2p comparison; None = auto
YLIM_LAT  = (0, 30)     # y-range for latency plots (ms)

RUNS    = [load_run(f) for f in CSVS]
muscles = [c for c in RUNS[0][2] if c != "Trigger A" and all(c in r[2] for r in RUNS)]
for lab, (meta_, t_, sig_) in zip(LABELS, RUNS):
    print(f"{lab:16s} electrode {meta_[0]['electrode']} | pw {meta_[0]['pw_us']} us | "
          f"intensities {[x['amp_ma'] for x in meta_]} mA")

def _latfile(csv):
    return f"results/latency_{os.path.splitext(os.path.basename(csv))[0]}.csv"
for lab, csv in zip(LABELS, CSVS):
    f = _latfile(csv)
    print(f"{lab:16s} " + ("picks saved:  " if os.path.exists(f) else "NO PICKS YET: ") + f)


## 2 · Manual latency picking — one block per file, do each ONCE

Every figure below reads the picks from `results/latency_<file>.csv`; the config cell above says
which conditions still say **NO PICKS YET**. Each condition has its own three cells:

- **A** — loads that file (and any picks already saved for it). Always safe to run.
- **B** — the picker: **uncomment**, run, click the response onset on every muscle × intensity
  (**Set NaN** = no response · ◀ Prev / Next ▶ · muscle dropdown), then comment it again.
- **C** — saves the picks to that file's own CSV: **uncomment**, run, comment again.

The cells of one block only talk to each other (`picks_<name>`), so running the blocks in any
order, or the rest of the notebook in between, can't mix conditions up. A save is refused if
the picks don't match the file's intensities.

### 2·pre · cathodic

In [ ]:
# ---- pre · cathodic · A: load ----
CSV_pre_cat = D + FILES["cathodic"][0]
meta_pre_cat, t_pre_cat, sig_pre_cat = load_run(CSV_pre_cat)
picks_pre_cat = load_latency_csv(_latfile(CSV_pre_cat)) if os.path.exists(_latfile(CSV_pre_cat)) else None
print("pre · cathodic:", CSV_pre_cat.split("/")[-1], "|", [m["amp_ma"] for m in meta_pre_cat], "mA")
print("existing picks reloaded - continue/correct them" if picks_pre_cat else "no picks yet - pick them in B")


In [ ]:
# ---- pre · cathodic · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_pre_cat = latency_picker(meta_pre_cat, t_pre_cat, sig_pre_cat, muscles, xlim=XLIM, manual_peaks=picks_pre_cat)


In [ ]:
# ---- pre · cathodic · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_pre_cat, muscles, CSV_pre_cat, meta=meta_pre_cat))


### 2·pre · anodic

In [ ]:
# ---- pre · anodic · A: load ----
CSV_pre_an = D + FILES["anodic"][0]
meta_pre_an, t_pre_an, sig_pre_an = load_run(CSV_pre_an)
picks_pre_an = load_latency_csv(_latfile(CSV_pre_an)) if os.path.exists(_latfile(CSV_pre_an)) else None
print("pre · anodic:", CSV_pre_an.split("/")[-1], "|", [m["amp_ma"] for m in meta_pre_an], "mA")
print("existing picks reloaded - continue/correct them" if picks_pre_an else "no picks yet - pick them in B")


In [ ]:
# ---- pre · anodic · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_pre_an = latency_picker(meta_pre_an, t_pre_an, sig_pre_an, muscles, xlim=XLIM, manual_peaks=picks_pre_an)


In [ ]:
# ---- pre · anodic · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_pre_an, muscles, CSV_pre_an, meta=meta_pre_an))


### 2·post · cathodic

In [ ]:
# ---- post · cathodic · A: load ----
CSV_post_cat = D + FILES["cathodic"][1]
meta_post_cat, t_post_cat, sig_post_cat = load_run(CSV_post_cat)
picks_post_cat = load_latency_csv(_latfile(CSV_post_cat)) if os.path.exists(_latfile(CSV_post_cat)) else None
print("post · cathodic:", CSV_post_cat.split("/")[-1], "|", [m["amp_ma"] for m in meta_post_cat], "mA")
print("existing picks reloaded - continue/correct them" if picks_post_cat else "no picks yet - pick them in B")


In [ ]:
# ---- post · cathodic · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_post_cat = latency_picker(meta_post_cat, t_post_cat, sig_post_cat, muscles, xlim=XLIM, manual_peaks=picks_post_cat)


In [ ]:
# ---- post · cathodic · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_post_cat, muscles, CSV_post_cat, meta=meta_post_cat))


### 2·post · anodic

In [ ]:
# ---- post · anodic · A: load ----
CSV_post_an = D + FILES["anodic"][1]
meta_post_an, t_post_an, sig_post_an = load_run(CSV_post_an)
picks_post_an = load_latency_csv(_latfile(CSV_post_an)) if os.path.exists(_latfile(CSV_post_an)) else None
print("post · anodic:", CSV_post_an.split("/")[-1], "|", [m["amp_ma"] for m in meta_post_an], "mA")
print("existing picks reloaded - continue/correct them" if picks_post_an else "no picks yet - pick them in B")


In [ ]:
# ---- post · anodic · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_post_an = latency_picker(meta_post_an, t_post_an, sig_post_an, muscles, xlim=XLIM, manual_peaks=picks_post_an)


In [ ]:
# ---- post · anodic · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_post_an, muscles, CSV_post_an, meta=meta_post_an))


## 3 · Raw traces — waterfalls

One trace per intensity stacked at its amplitude, red = stimulus artifact. **Same gain per muscle
in both** (the first call returns the gains, the second reuses them), so a smaller response
with lidocaine actually draws smaller.

### 3a · Before lidocaine

In [ ]:
gains = waterfall(*RUNS[0], muscles, xlim=XLIM)          # first condition sets the gain


### 3b · With lidocaine — same gain

In [ ]:
for lab, (meta_, t_, sig_) in zip(LABELS[1:], RUNS[1:]):
    print(lab); waterfall(meta_, t_, sig_, muscles, xlim=XLIM, gains=gains)


### 3c · Overlay — before and with lidocaine on the same panels, same gain

In [ ]:
waterfall_overlay(RUNS, muscles=muscles, xlim=XLIM, gains=gains, labels=LABELS);


## 4 · Check — which peaks the peak-to-peak uses

Per muscle, every intensity, the max (red ▲) and min (blue ▼) inside the window that starts at
the picked latency. If a marker sits on the artifact or misses the wave, fix the pick (§2) or
`WINDOW_MS` / `OFFSET_MS` (§1).

### 4a · Each condition (files without picks are skipped)

In [ ]:
for lab, csv, (meta_, t_, sig_) in zip(LABELS, CSVS, RUNS):
    if not os.path.exists(_latfile(csv)):
        print(f"{lab}: no picks yet - skipped"); continue
    print(lab)
    plot_p2p_markers(meta_, t_, sig_, muscles, load_latency_csv(_latfile(csv)),
                     window_ms=WINDOW_MS, offset_ms=OFFSET_MS)


## 5 · Latency — before vs with lidocaine

One panel per muscle, x = intensity. **Gray = before, orange = with lidocaine; ● solid = left arm, ▲ dashed = right arm.**  Trapezius is right-only.

In [ ]:
compare_per_muscle(CSVS, None, metric="latency", ylim=YLIM_LAT, labels=LABELS);


## 6 · Peak-to-peak — before vs with lidocaine

Same layout, p2p in mV (window `[latency + OFFSET_MS, + WINDOW_MS]`). y-axis is per muscle —
biceps is ~10× the thenar — so compare gray vs orange within a panel, not heights across panels.

In [ ]:
compare_per_muscle(CSVS, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=LABELS);


## 7 · The other polarity — before vs with lidocaine

In [ ]:
CSVS_O = [D + f for f in FILES[OTHER]]
compare_per_muscle(CSVS_O, None, metric="latency", ylim=YLIM_LAT, labels=[f"{OTHER} · {l}" for l in LABELS]);
compare_per_muscle(CSVS_O, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=[f"{OTHER} · {l}" for l in LABELS]);


## 8 · Both polarities together

Gray / orange = cathodic before / with lidocaine, blue / green = anodic before / with lidocaine.

In [ ]:
LABELS4 = ["cathodic · before", "cathodic · lidocaine", "anodic · before", "anodic · lidocaine"]
CSVS4   = [D + FILES["cathodic"][0], D + FILES["cathodic"][1], D + FILES["anodic"][0], D + FILES["anodic"][1]]
compare_per_muscle(CSVS4, None, metric="latency", ylim=YLIM_LAT, labels=LABELS4);
compare_per_muscle(CSVS4, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=LABELS4);
